# Imports & Functions

In [312]:
import pandas as pd
import numpy as np
from functools import partial
import optuna.visualization as vis

from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sklearn.ensemble import HistGradientBoostingRegressor
import xgboost as xgb
import optuna


In [313]:
def remove_top_1_percent_outliers(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    df_clean = df.copy()
    for col in features:
        lower_bound = np.percentile(df_clean[col], 1)
        upper_bound = np.percentile(df_clean[col], 99)
        
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    
    return df_clean

def remove_outliers(df, target, threshold=0.012):
    """Removes the extreme outliers"""
    lower_bound = target.quantile(threshold)
    upper_bound = target.quantile(1 - threshold)
    
    mask = (target >= lower_bound) & (target <= upper_bound)
    return df[mask], target[mask]

def frequency_encoding(df_to_modify: pd.DataFrame, df_initial: pd.DataFrame, column: str) -> pd.DataFrame:
    freq_map = df_initial[column].value_counts()
    df_to_modify[column + "_freq"] = df_initial[column].map(freq_map).fillna(0)
    return df_to_modify

# Function to optimize the parameters of the XGBoost
def objective(trial, X_train_clean, Y_train_clean, X_val, y_val):
    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
        "objective": "reg:squarederror",
        "eval_metric": "mae"
    }

    dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
    dval = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    preds = model.predict(dval)
    preds = preds.round().astype(int)
    mae = mean_absolute_error(y_val, preds)
    return mae


# Read files

In [314]:
x_test_file = pd.read_csv(r"x_test_final.csv")
x_train_file = pd.read_csv(r"x_train_final.csv")
y_sample = pd.read_csv(r"y_sample_final.csv")
y_train_file = pd.read_csv(r"y_train_final_j5KGWWK.csv")

# Data & Engineering

### Train Data

In [315]:
###### Train through Sklearn #####

x_train, x_val, y_train, y_val = train_test_split(x_train_file, y_train_file, test_size=0.1, random_state=54)
y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

In [316]:
##### Train through simple way #####

split_idx = int(len(x_train_file) * 0.9)
x_train, x_val = x_train_file.iloc[:split_idx], x_train_file.iloc[split_idx:]
y_train, y_val = y_train_file.iloc[:split_idx], y_train_file.iloc[split_idx:]

y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

### Outliers & Encoding

In [317]:
###################################
##### OLD VERSION OF OUTLIERS #####
###################################

outlier_features = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
x_train = remove_top_1_percent_outliers(x_train, outlier_features)
y_train = y_train.loc[x_train.index]

In [318]:
##### Encoding #####

features_to_keep = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
X_train = x_train[features_to_keep].copy()
X_val = x_val[features_to_keep].copy()
X_test = x_test_file[features_to_keep].copy()
Y_train = y_train.copy()

# Feature Engineering: Encode "gare" and "arret" column
gare_counts = x_train['gare'].value_counts()
X_train['gare_encoded'] = x_train['gare'].map(gare_counts)
X_val['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)
X_test['gare_encoded'] = x_test_file['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X_train['arret_encoded'] = x_train['arret'].map(arret_counts)
X_val['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)
X_test['arret_encoded'] = x_test_file['arret'].map(arret_counts).fillna(0)

In [319]:
##### Remove Outliers #####

X_train_clean, Y_train_clean = remove_outliers(X_train, Y_train, threshold=0.012)

# Ensure validation set has same columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Weighted Average Model

In [320]:
## First method with weighted mean

x_test_final_test = x_test_file.copy()
x_test_final_test["p0q0"] = (0.6 * x_test_final_test["p0q2"] + 0.3 * x_test_final_test["p0q3"] + 0.1 * x_test_final_test["p0q4"]).round(0)
y_first_test = x_test_final_test["p0q0"]
y_first_test.to_csv("Weighted_Average.csv")

# Random Forest Model

In [321]:
# Initialise the model
rf = RandomForestRegressor(n_estimators=100, random_state=54)

# Train the model
rf.fit(X_train, Y_train.values.ravel())

# Predecit on all values
y_pred = rf.predict(X_val)

# Evaluate the model
mae = mean_absolute_error(y_val, y_pred)
print(f"Erreur absolue moyenne (MAE) : {mae}")


Erreur absolue moyenne (MAE) : 0.8281826884101715


In [322]:
y_test_pred = rf.predict(X_test)
submission = pd.DataFrame({'p0q0': y_test_pred})
submission.to_csv("submission_RF.csv", index=False)

# HGB Model

In [323]:
# Boosted gradient model
hgb = HistGradientBoostingRegressor(
    max_iter=1000,
    max_depth=50,
    learning_rate=0.2,
    max_bins=255,
    l2_regularization=1,
    random_state=42)

hgb.fit(X_train, Y_train)

y_pred = hgb.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
print(f"Erreur absolue moyenne (MAE) : {mae}")

Erreur absolue moyenne (MAE) : 0.7492288303850064


In [324]:
# Predict the test file
y_test_pred = hgb.predict(X_test)

# Create the good file
submission = pd.DataFrame({
    "Unnamed: 0": y_sample["Unnamed: 0"],
    "p0q0": y_test_pred
})

submission.to_csv("submission_hgb.csv", index=False)

# XGBoost Model

In [325]:
# Initialize the model
xgb_model = xgb.XGBRegressor(
    n_estimators=1400,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.009,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    reg_alpha=1,      # L1 (lasso)
    reg_lambda=2,     # L2 (ridge)
    random_state=54,
    n_jobs=-1           # Use all CPU cores
)

# Fit the model
xgb_model.fit(
    X_train_clean,
    Y_train_clean,
)

# Predict
y_pred = xgb_model.predict(X_val)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)

# Evaluate
mae = mean_absolute_error(y_val, y_pred)
print(f"Erreur absolue moyenne (MAE) avec XGBoost : {mae}")


Erreur absolue moyenne (MAE) avec XGBoost : 0.6664168927120956


In [326]:
# Predict on test
y_test_pred = xgb_model.predict(X_test)
y_test_pred = y_test_pred.round(0).astype(int)
y_test_pred = pd.DataFrame(y_test_pred)

# Save in a csv file
y_test_pred.to_csv("submission_xgboost9.csv")

print("Fichier de soumission sauvegardé : submission_xgboost9.csv")


Fichier de soumission sauvegardé : submission_xgboost9.csv


# XGBoost Model with optimized parameters

### Search the best parameters an plot

In [327]:
objective_with_data = partial(
    objective,
    X_train_clean=X_train_clean,
    Y_train_clean=Y_train_clean,
    X_val=X_val,
    y_val=y_val
)

study = optuna.create_study(direction="minimize")
study.optimize(objective_with_data, n_trials=30)

print("🎯 Meilleurs hyperparamètres trouvés :")
print(study.best_params)

[I 2025-03-30 21:23:38,599] A new study created in memory with name: no-name-4f86dc3c-e58c-42d4-be99-df446d5404eb
[I 2025-03-30 21:23:50,325] Trial 0 finished with value: 0.665487733601091 and parameters: {'max_depth': 10, 'learning_rate': 0.021222072855851853, 'subsample': 0.8914217819309884, 'colsample_bytree': 0.7803410935195356, 'reg_alpha': 3.043805725735862, 'reg_lambda': 0.5499338700853018}. Best is trial 0 with value: 0.665487733601091.
[I 2025-03-30 21:23:56,943] Trial 1 finished with value: 0.6664318791493699 and parameters: {'max_depth': 9, 'learning_rate': 0.038699594446456036, 'subsample': 0.869483304439088, 'colsample_bytree': 0.8778914235911064, 'reg_alpha': 3.654681505795091, 'reg_lambda': 0.7703957784978466}. Best is trial 0 with value: 0.665487733601091.
[I 2025-03-30 21:24:10,556] Trial 2 finished with value: 0.6658174352211249 and parameters: {'max_depth': 15, 'learning_rate': 0.01521073004456858, 'subsample': 0.8330583310410125, 'colsample_bytree': 0.61159351460185

🎯 Meilleurs hyperparamètres trouvés :
{'max_depth': 12, 'learning_rate': 0.02317124079856121, 'subsample': 0.6080274587027206, 'colsample_bytree': 0.6411875452392144, 'reg_alpha': 2.6669288089076435, 'reg_lambda': 0.9043248072047865}


In [328]:
## Plot the results of the parameters optimization

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

### Train the model

In [329]:
best_params = study.best_params
best_params.update({
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "n_jobs": -1,
    "random_state": 42
})

dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
dval = xgb.DMatrix(X_val, label=y_val)

best_xgboost_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, "validation")],
    early_stopping_rounds=50,
    verbose_eval=True,
)

y_pred = best_xgboost_model.predict(dval).round().astype(int)
mae = mean_absolute_error(y_val, y_pred)
print(f"MAE avec les meilleurs paramètres : {mae:.4f}")


[0]	validation-mae:0.90595
[1]	validation-mae:0.89881
[2]	validation-mae:0.89331
[3]	validation-mae:0.88807
[4]	validation-mae:0.88319
[5]	validation-mae:0.87976
[6]	validation-mae:0.87642
[7]	validation-mae:0.87363
[8]	validation-mae:0.86901
[9]	validation-mae:0.86659
[10]	validation-mae:0.86366
[11]	validation-mae:0.85960
[12]	validation-mae:0.85620
[13]	validation-mae:0.85384
[14]	validation-mae:0.85035
[15]	validation-mae:0.84720
[16]	validation-mae:0.84506
[17]	validation-mae:0.84305
[18]	validation-mae:0.84132
[19]	validation-mae:0.83796
[20]	validation-mae:0.83547
[21]	validation-mae:0.83374
[22]	validation-mae:0.83096
[23]	validation-mae:0.82927
[24]	validation-mae:0.82733
[25]	validation-mae:0.82467
[26]	validation-mae:0.82233
[27]	validation-mae:0.82091
[28]	validation-mae:0.81865
[29]	validation-mae:0.81612
[30]	validation-mae:0.81462
[31]	validation-mae:0.81315
[32]	validation-mae:0.81168
[33]	validation-mae:0.80941
[34]	validation-mae:0.80753
[35]	validation-mae:0.80557
[3

### Create the submission file

In [330]:
X_full = pd.concat([X_train_clean, X_val], axis=0)
Y_full = pd.concat([Y_train_clean, y_val], axis=0)
dtrain_full = xgb.DMatrix(X_full, label=Y_full)
X_test = X_test.reindex(columns=X_train_clean.columns, fill_value=0)
dtest = xgb.DMatrix(X_test)

final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=best_xgboost_model.best_iteration + 50  # 🔁 Nombre optimal trouvé + marge
)

# Prédictions
y_test_pred = final_model.predict(dtest)
y_test_pred = y_test_pred.round(0).astype(int)

# Convertir en DataFrame avec index (comme y_sample)
y_test_pred = pd.DataFrame(y_test_pred, columns=["p0q0"])
y_test_pred.index.name = "index"

# Sauvegarde dans un CSV
y_test_pred.to_csv("submission_xgboost12.csv")

print("Fichier de soumission sauvegardé : submission_xgboost12.csv")


Fichier de soumission sauvegardé : submission_xgboost12.csv
